In [ ]:
%load_ext autoreload
%autoreload 2

In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# stats packages
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols, mixedlm
from statsmodels.stats.anova import AnovaRM

import scipy.optimize

# set save loaction for figures
fig_save_path = ""

In [ ]:
cd "/app/"

In [ ]:
%run env.py
%run run.py connect

In [22]:
from vr4mice.schema import vr4mice, dlc, base_analysis
import vr4mice.analysis.plotting as plotting
import vr4mice.analysis.utils as utils
import vr4mice.analysis.analysis as analysis
import vr4mice.analysis.regression as regression
analysis.style()


In [23]:
colors_choice = ["#5C0A72", "#FD672C"]
colors_aperture = ["#E41A1C", "#437FB5", "#4daf4a", "#984ea3", "#ff7f00"]
colors_aperture_pale = ["#EC8788", "#96B9D6"]

# Compute data

In [ ]:
for d in vr4mice.Metadata().fetch("dataset", "session_label", as_dict=True): 
    if "ar_shape_discrimination" in d["session_label"]:
        print({"dataset": d["dataset"]}, ",")

In [10]:
list = [{'dataset': 'Uguisu_2024-09-06_1'} ,
{'dataset': 'Uguisu_2024-09-09_1'} ,
{'dataset': 'Uguisu_2024-09-10_1'} ,
{'dataset': 'Uguisu_2024-09-11_1'} ,
{'dataset': 'Uguisu_2024-09-12_1'} ,
{'dataset': 'Vegavis_2024-09-05_2'} ,
{'dataset': 'Vegavis_2024-09-06_1'} ,
{'dataset': 'Vegavis_2024-09-10_1'} ,
{'dataset': 'Vegavis_2024-09-11_1'} ,
{'dataset': 'Vegavis_2024-09-12_1'} ,
{'dataset': 'Vegavis_2024-09-13_1'} ,
{'dataset': 'Vegavis_2024-09-16_1'} ,
{'dataset': 'Wiwaxia_2024-09-10_1'} ,
{'dataset': 'Wiwaxia_2024-09-11_1'} ,
{'dataset': 'Wiwaxia_2024-09-12_1'} ,
{'dataset': 'Wiwaxia_2024-09-16_1'} ,
{'dataset': 'Wiwaxia_2024-09-17_1'} ,
{'dataset': 'Wiwaxia_2024-09-18_1'} ,
{'dataset': 'Wiwaxia_2024-09-19_1'} ,
{'dataset': 'Wiwaxia_2024-09-20_1'} ,
{'dataset': 'Wiwaxia_2024-09-21_1'} ,]

In [11]:
def get_all_in_list(data_set_list, training_stage):
    print(training_stage)
    big_df = []
    
    for d in data_set_list:
        split_d = d["dataset"].split("_")
        print(split_d)
        
        offline_kinematics_df = dlc.OfflineKinematics().get_data(key=d)
        df = base_analysis.DataFrame().get_data(key=d, 
                                                        columns=[
                                                            'dataset', 'trial', 'aperture',
                                                            'trial_right_choice', 'trial_left_choice',
                                                            'velocity', 'velocity_x', 'velocity_y',
                                                            "reward",
                                                            'norm_y', "iti", "x", "y",
                                                            'trial_init_x', 'trial_init_y',
                                                            "trial_tortuosity", "trial_duration"
        ])
        df["trial_rewarded"] = base_analysis.DataFrame().get_rewarded(key=d)
        
        df = df.join(offline_kinematics_df)
        
        df["mouse_name"] = split_d[0]
        df["date"] = split_d[1]
        df["attempt"] = split_d[2]
        df["training_stage"] = training_stage
        
        big_df.append(df)
    
    big_df =  pd.concat(big_df).reset_index()
    big_df ["session_increment"] = np.array(big_df.groupby("dataset").ngroup()+1)
    big_df = big_df.infer_objects()
    
    return(big_df.reset_index(drop=True))

In [ ]:
big_df = get_all_in_list(data_set_list = list, training_stage="discrimination")

In [83]:
big_df.to_pickle("big_df_multi_occluded_shape.pkl")

# Start of the analyses

In [111]:
big_df = pd.read_pickle("big_df_multi_occluded_shape.pkl")

In [ ]:
big_df = big_df[~(big_df.dataset.isin(['Uguisu_2024-09-20_1', 'Vegavis_2024-09-20_1', "Uguisu_2024-09-25_1",
                                       'Uguisu_2024-09-13_1']))]

#big_df = big_df[~(big_df.mouse_name=="Vegavis")]
big_df = big_df[big_df.iti == 0]
big_df.dataset.unique()

In [127]:
session_to_plot =  'Uguisu_2024-10-02_1'

In [ ]:
fig,ax = plt.subplots(1,1, figsize = (5,5))
plotting.plot_rewards(df=big_df, ax=ax, alpha=0.2, per_aperture=True, per_mouse=True, cmap="Set2")
#plotting.plot_rewards(df=big_df, ax=ax[1], alpha=0.3, per_aperture=True, per_mouse=False)
plt.ylim(0,1)
#plt.xlim(-0.5,1.5)
sns.despine(offset=10)
#plt.savefig(fig_save_path + "dual_occluder_rewards.svg", transparent=True)

In [ ]:
plotting.plot_tortuosity_duration_distribution(big_df, log_scale=True)

In [130]:
box_df = base_analysis.BoxDataFrame().get_data(key={"dataset": 'Uguisu_2024-09-23_1'})

In [ ]:
plotting.plot_session(
    df=big_df[big_df.dataset == session_to_plot],
    box_df=box_df,
    per_aperture=True,
    per_side=True,
)

In [132]:
big_df["optimal_p"] = analysis.get_optimal_p(big_df)
big_df["local_tortuosity"] = analysis.get_local_tortuosity(big_df, window_size=1)
big_df["flip_one_side"] = big_df["trial_left_choice"].replace([0, 1], [1, -1])
big_df["distance_to_choice"] = analysis.get_distance_to_choice(big_df, box_df)

j_shaped = analysis.get_jshaped_trials(big_df)

In [ ]:
plotting.plot_session(
    df=j_shaped[j_shaped.dataset == session_to_plot],
    box_df=box_df,
    per_aperture=True,
    per_side=True,
)

In [ ]:
j_shaped = utils.create_bins(data=j_shaped, spatial_ybins=[0, 20, 35], label="y")
mean_mouse = j_shaped.groupby(
    ["dataset", "trial_right_choice", "aperture", "bin_centers"], as_index=False
).mean(numeric_only=True)

#mean_mouse = analysis.get_optimal_p()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

plotting.lineplot_flip_axis(
    data=mean_mouse,
    x="bin_centers",
    y="x",
    hue="trial_right_choice" if len(mean_mouse.aperture.unique()) == 2 else "aperture",
    palette=colors_choice if len(mean_mouse.aperture.unique()) == 2 else "viridis",
    style="aperture" if len(mean_mouse.aperture.unique()) == 2 else "trial_right_choice",
    errorbar="se",
    ax=ax,
)
ax.set_ylabel("y position (cm)")
ax.set_xlabel("x position (cm)")

ax.set_xlim(-15, 15)
plt.legend(loc="lower right")

plt.show()


In [ ]:
fig, ax = plt.subplots(5, 4, figsize=(25, 25))
ax = ax.flatten()

df_init = big_df

hists = []
for i, session in enumerate(df_init.dataset.unique()):
    hist = plotting.plot_init_position_histogram(
        df_init[df_init.dataset == session],
        box_df,
        ax=ax[i],
        bins=3,
        cmap="magma",
        vmax=40,
        is_density=False,
    )

    number_trials = df_init[df_init.dataset == session].groupby(["trial"]).x.first().count()
    hist = hist / number_trials
    hists.append(hist)

    ax[i].set_title(f"{session}")
plt.close()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

sns.heatmap(
    np.flip(np.mean(hists, axis=0), axis=1).T,
    cmap="magma",
    annot=(np.flip(np.mean(hists, axis=0), axis=1).T),
    vmax=0.15,
)

ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
fig, ax = plt.subplots(4, 3, figsize=(25, 28))
ax = ax.flatten()

df = big_df 

hists = []
for i, session in enumerate(df.dataset.unique()):
    hist = plotting.plot_init_position_histogram(
        df[df.dataset == session],
        box_df,
        ax=ax[i],
        bins=10,
        cmap="magma",
        vmax=10,
        is_density=False,
    )

    number_trials = df[df.dataset == session].groupby(["trial"]).x.first().count()
    hist = hist / number_trials
    hists.append(hist)

    ax[i].set_title(f"{session}")
    plotting.plot_all_boxes(ax=ax[i], box_df=box_df)

plt.tight_layout(pad=2.0)

In [137]:
columns = [
    "y",
    "heading_dir",
    "head_angle",
    "trial_tortuosity",
    "local_tortuosity",
    "trial_duration",
    "x",
    "aperture",
    "velocity",
    "velocity_x",
    "velocity_y",
    "trial_rewarded",
    "norm_y",
    "flip_one_side",
    "distance_to_choice",
    "optimal_p"
]

interpolated_j_shaped = utils.interpolate(
    j_shaped, n_points=200, value_columns=["trial_left_choice"] + columns
)

interpolated_j_shaped["trial_step"] = interpolated_j_shaped.groupby(
    ["dataset", "trial"]
).trial.cumcount()
interpolated_j_shaped["trial_length"] = interpolated_j_shaped["trial_step"] / 200

interpolated_j_shaped["velocity_x_fliped"] = (
    interpolated_j_shaped["velocity_x"] * interpolated_j_shaped["flip_one_side"]
)

interpolated_j_shaped["head_angle_fliped"] = (
    interpolated_j_shaped["head_angle"] * interpolated_j_shaped["flip_one_side"]
)

interpolated_j_shaped["heading_dir_fliped"] = (
    interpolated_j_shaped["heading_dir"] * interpolated_j_shaped["flip_one_side"]
)
#interpolated_j_shaped["heading_dir_re"] = interpolated_j_shaped["heading_dir"] - 90

In [ ]:
mean_mouse = interpolated_j_shaped.groupby(
    ["dataset", "aperture", "trial_length"], as_index=False
).mean(numeric_only=True)

fig, ax = plt.subplots(3, 2, figsize=(12, 12))
ax = ax.flatten()

for i, label in enumerate(
    ["velocity_x_fliped", "velocity_y", "velocity", "local_tortuosity", "distance_to_choice"]
):
    sns.lineplot(
        data=mean_mouse,
        x="trial_length",
        y=label,
        palette=(
            colors_aperture[:2]
            if len(mean_mouse.aperture.unique()) == 2
            else "viridis"
        ),
        hue="aperture",
        errorbar="se",
        ax=ax[i],
    )
    ax[i].set_title(f"{label}")
plt.tight_layout(pad=2)

In [ ]:
mean_mouse = interpolated_j_shaped.groupby(
    ["dataset", "aperture", "trial_rewarded", "trial_length"], as_index=False
).mean(numeric_only=True)

fig, ax = plt.subplots(3, 2, figsize=(12, 12))
ax = ax.flatten()

for i, label in enumerate(
    ["velocity_x_fliped", "velocity_y", "velocity", "local_tortuosity", "distance_to_choice"]
):
    sns.lineplot(
        data=mean_mouse,
        x="trial_length",
        y=label,
        palette=(
            colors_aperture[:2]
            if len(mean_mouse.aperture.unique()) == 2
            else "viridis"
        ),
        hue="aperture",
        style="trial_rewarded",
        errorbar="se",
        ax=ax[i],
    )
    ax[i].set_title(f"{label}")
plt.tight_layout(pad=2)

In [ ]:
fig, ax = plt.subplots(1, 1)

counts = j_shaped[j_shaped.trial_rewarded==1].groupby(["mouse_name", "dataset", "aperture"], as_index=False).mean()

counts["count"] = counts["optimal_p"]
counts = pd.DataFrame(counts.reset_index())
counts.aperture = counts.aperture.round(2).astype(str)

plotting._plot_bar_counts(
    counts=counts,
    label_x="aperture",
    alpha=0.2,
    ax=ax,
    per_mouse=True,
    cmap="Set2",
)
ax.invert_xaxis()
ax.set_ylabel("Optimal p")


for i in counts.aperture.unique():
    for j in counts.aperture.unique():
        if i < j:
            stat = stats.ttest_rel(
                counts[counts["aperture"] == i]["optimal_p"],
                counts[counts["aperture"] == j]["optimal_p"],
            )
            print(f"{i}-{j}: {stat}")

In [ ]:
fig, ax = plt.subplots(1, 1)

counts = big_df.groupby(["mouse_name", "dataset", "aperture"], as_index=False).mean()

counts["count"] = counts["local_tortuosity"]
counts = pd.DataFrame(counts.reset_index())
counts.aperture = counts.aperture.round(2).astype(str)

plotting._plot_bar_counts(
    counts=counts,
    label_x="aperture",
    alpha=0.2,
    ax=ax,
    per_mouse=True,
    cmap="Set2",
)
ax.invert_xaxis()
ax.set_ylabel("Mean local tortuosity")


for i in counts.aperture.unique():
    for j in counts.aperture.unique():
        if i < j:
            stat = stats.ttest_rel(
                counts[counts["aperture"] == i]["trial_tortuosity"],
                counts[counts["aperture"] == j]["trial_tortuosity"],
            )
            print(f"{i}-{j}: {stat}")

In [ ]:
fig, ax = plt.subplots(1, 1)

counts = big_df.groupby(["dataset", "trial"], as_index=False).first().groupby(["mouse_name", "dataset", "aperture"], as_index=False).mean()

counts["count"] = counts["trial_tortuosity"]
counts = pd.DataFrame(counts.reset_index())
counts.aperture = counts.aperture.round(2).astype(str)

plotting._plot_bar_counts(
    counts=counts,
    label_x="aperture",
    per_day=False,
    alpha=0.2,
    ax=ax,
    per_mouse=True, 
    cmap="Set2",
)
ax.invert_xaxis()
ax.set_ylabel("Mean trial tortuosity")


for i in counts.aperture.unique():
    for j in counts.aperture.unique():
        if i < j:
            stat = stats.ttest_rel(
                counts[counts["aperture"] == i]["trial_tortuosity"],
                counts[counts["aperture"] == j]["trial_tortuosity"],
            )
            print(f"{i}-{j}: {stat}")


In [ ]:
mean_mouse = interpolated_j_shaped.groupby(
    ["dataset", "aperture", "trial_left_choice", "trial_length"], as_index=False
).mean(numeric_only=True)

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax = ax.flatten()

for i, label in enumerate(["heading_dir", "head_angle", "velocity_x"]):
    sns.lineplot(
        data=mean_mouse,
        x="trial_length",
        y=label,
        palette=colors_choice if len(mean_mouse.aperture.unique()) == 2 else "viridis",
        hue="trial_left_choice"
        if len(mean_mouse.aperture.unique()) == 2
        else "aperture",
        style=(
            "aperture"
            if len(mean_mouse.aperture.unique()) == 2
            else "trial_left_choice"
        ),
        errorbar="se",
        ax=ax[i],
    )
    ax[i].set_title(f"{label}")
    
        
plt.tight_layout(pad=2)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(20, 10))

for j in [0, 1]:
    
    mean_mouse = interpolated_j_shaped[interpolated_j_shaped.trial_rewarded==j].groupby(
        ["dataset", "aperture", "trial_left_choice", "trial_length"], as_index=False
    ).mean(numeric_only=True)

    for i, label in enumerate(["heading_dir", "head_angle", "velocity_x"]):
        sns.lineplot(
            data=mean_mouse,
            x="trial_length",
            y=label,
            palette=colors_choice if len(mean_mouse.aperture.unique()) == 2 else "viridis",
            hue="trial_left_choice"
            if len(mean_mouse.aperture.unique()) == 2
            else "aperture",
            style=(
                "aperture"
                if len(mean_mouse.aperture.unique()) == 2
                else "trial_left_choice"
            ),
            errorbar="se",
            ax=ax[j][i],
        )
        ax[j][i].set_title(f"{label}_rewarded:{j}")
        
        # if i==1:
        #     ax[j][i].set_ylim(-20, 50)
        # elif i==2:
        #     ax[j][i].set_ylim(-40, 40)
            
plt.tight_layout(pad=2)

In [ ]:
j_shaped = utils.create_bins(data=j_shaped, spatial_ybins=[6.25, 20, 35], label="y")
mean_mouse = j_shaped.groupby(
    ["dataset", "aperture", "bin_centers"], as_index=False
).mean(numeric_only=True)

sns.lineplot(data=mean_mouse,
             x="bin_centers",
             y="local_tortuosity",
             hue="aperture",
             errorbar="se",
             palette="viridis")


## Regression model

In [146]:
big_df["norm_x"] = big_df.groupby(["dataset", "trial"], as_index=False)["y"].transform(
        lambda x: x - np.mean(x.iloc[:3])
    )

In [147]:
columns = [
    "norm_y",
    "norm_x",
    "heading_dir",
    "head_angle",
    "trial_tortuosity",
    "local_tortuosity",
    "trial_duration",
    "x",
    "y",
    "aperture",
    "velocity",
    "velocity_x",
    "velocity_y",
    "trial_rewarded",
    "norm_y",
    "flip_one_side",
    "distance_to_choice",
    "optimal_p"
]
j_shaped = analysis.get_jshaped_trials(big_df)

#j_shaped = j_shaped[j_shaped["trial_rewarded"]==1]

n_samples = 500
interpolated_j_shaped = utils.interpolate(
    j_shaped, n_points=n_samples, value_columns=["trial_left_choice"] + columns
)
interpolated_j_shaped["trial_step"] = interpolated_j_shaped.groupby(
    ["mouse_name", "dataset", "trial"], as_index=False).trial.cumcount()


interpolated_j_shaped["trial_length"] = interpolated_j_shaped["trial_step"] / n_samples
interpolated_j_shaped["head_angle_sin"] = np.sin(np.deg2rad(interpolated_j_shaped.head_angle))
interpolated_j_shaped["head_angle_cos"] = np.cos(np.deg2rad(interpolated_j_shaped.head_angle))

interpolated_j_shaped["heading_dir_sin"] = np.sin(np.deg2rad(interpolated_j_shaped.heading_dir))
interpolated_j_shaped["heading_dir_cos"] = np.cos(np.deg2rad(interpolated_j_shaped.heading_dir))

interpolated_j_shaped["velocity_x_fliped"] = (
    interpolated_j_shaped["velocity_x"] * interpolated_j_shaped["flip_one_side"]
)

In [148]:
model_labels = [
    "x",
    "y",
    "velocity_x",
    "velocity_y",
    "heading_dir_sin",
    "heading_dir_cos",
    "head_angle_sin",
    "head_angle_cos",
    "trial_tortuosity",
    "local_tortuosity",
    "trial_duration",
    "aperture",
    "trial_rewarded",
    "trial_length",
    "distance_to_choice",
    "optimal_p"
]

In [162]:
interpolated_j_shaped["aperture"] = interpolated_j_shaped["aperture"].astype(float)
interpolated_j_shaped = interpolated_j_shaped[interpolated_j_shaped.trial_rewarded==1]

df_model, coef = regression.predict_decision(
    df=interpolated_j_shaped, label=model_labels, per_mouse=True,
    scale_data = True, max_iter=500)

In [ ]:
fig, ax = plt.subplots(1, len(interpolated_j_shaped.aperture.unique()), figsize=(15, 5))

trials = range(20,40)
group = df_model[(df_model.trial.isin(trials)) & (df_model.dataset == session_to_plot)]

for i, aperture in enumerate(interpolated_j_shaped.aperture.unique()):
    sns.lineplot(
        data=group[group.aperture==aperture],
        x="trial_length",
        y="proba_left",
        hue="trial",
        errorbar=None,
        estimator=None,
        units="trial",
        palette="viridis", #[plotting.colors_choice[1], plotting.colors_choice[0]],
        sort=False, alpha=0.5, ax=ax[i]
    )
    ax[i].set_ylabel("Proba(Left)")
    ax[i].set_xlabel("Trial progression")

In [ ]:
trials = range(20,80)
group = df_model[(df_model.trial.isin(trials)) & (df_model.dataset == session_to_plot)]

plotting.plot_session(
    df=group,
    box_df=box_df,
    per_aperture=True,
    per_side=True,
    scatter_reward=False
)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 8))

mean_mouse = df_model.groupby(["mouse_name", "dataset", "aperture", "trial_rewarded", "trial_length"]).mean(
    numeric_only=True
)

mean_mouse_per_choice = df_model.groupby(
    ["mouse_name", "dataset", "trial_left_choice", "aperture", "trial_length"]
).mean(numeric_only=True)

# Accuracy
sns.lineplot(
    ax=ax[0],
    data=mean_mouse,
    y="accuracy",
    x="trial_length",
    hue="aperture",
    style="trial_rewarded",
    palette=(
        colors_aperture[:2] if len(df_model.aperture.unique()) == 2 else "viridis"
    ),
    errorbar="se",
)
ax[0].hlines(xmin=0, xmax=1, y=0.5, linestyles="dashed", color="k")
ax[0].set_ylim(0.4, 1)

# P(Left)
sns.lineplot(
    ax=ax[1],
    data=mean_mouse_per_choice,
    y="proba_left",
    x="trial_length",
    hue="trial_left_choice" if len(df_model.aperture.unique()) == 2 else "aperture",
    style="aperture" if len(df_model.aperture.unique()) == 2 else "trial_left_choice",
    palette=(
        [colors_choice[1], colors_choice[0]]
        if len(df_model.aperture.unique()) == 2
        else "viridis"
    ),
)
ax[1].hlines(xmin=0, xmax=1, y=0.5, linestyles="dashed", color="k")
ax[1].hlines(xmin=0, xmax=1, y=0.8, linestyles="dotted", color="grey")
ax[1].hlines(xmin=0, xmax=1, y=0.2, linestyles="dotted", color="grey")

# Logits of the regression
ax[2].bar(
    model_labels, #np.arange(coef[:, 1:].shape[1]),
    np.mean(coef[:, 1:], axis=0),
    yerr=stats.sem(coef[:, 1:], axis=0),
)

ax[2].set_xticks(np.arange(len(model_labels)))  # Set the positions for the labels
ax[2].set_xticklabels(model_labels, rotation=45, ha="right")  # Rotate the labels

#ax[2].errorbar(np.arange(coef[:, 1:].shape[1]), np.mean(coef[:, 1:], axis=0), yerr=stats.sem(coef[:, 1:], axis=0), fmt='o', capsize=5, alpha=0.5)

plt.tight_layout(pad=1.0)
plt.show()

In [166]:
decision_points = regression.find_decision_point(df_model, threshold_uncertainty=0.3)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

parameter = "y"
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
diffs = []
for i in thresholds: # :
    decision_points = regression.find_decision_point(df_model, threshold_uncertainty=i)
    
    aperture_means = decision_points.groupby(["mouse_name", "dataset", "aperture"]).mean().groupby(["aperture"]).mean()[parameter]
    #aperture_means = decision_points.groupby(["mouse_name", "dataset", "aperture"]).mean().groupby(["aperture"]).mean()
    diffs.append(aperture_means.iloc[0] - aperture_means.iloc[-1])
    
ax.plot(thresholds, diffs)

In [ ]:
fig, ax = plt.subplots(1, len(df_model.aperture.unique()), figsize=(20, 5))

decision_color = "deeppink"

for i, aperture in enumerate(df_model.sort_values(by="aperture").aperture.unique()):
    regression.plot_decision_points_on_trajectory(
        df_model[
            (df_model.dataset == session_to_plot) & (df_model.aperture == aperture)
        ],
        box_df,
        decision_point=decision_points[(decision_points.dataset == session_to_plot) & (decision_points.aperture == aperture)],
        color=decision_color,
        trials=range(20, 80),
        ax=ax[i],
    )
    ax[i].set_title(f"{aperture}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plotting.pairplot_std_decision_point(decision_points, label_parameter="local_tortuosity", ax=ax[0], per_mouse=True, cmap="Set2")
plotting.pairplot_std_decision_point(decision_points, label_parameter="velocity_x", ax=ax[1], per_mouse=True, cmap="Set2")

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(10, 12))
ax = ax.flatten()
plotting.pairplot_average_decision_point(decision_points, label_parameter="y", ax=ax[0], per_mouse=True, cmap="Set2")
plotting.pairplot_average_decision_point(decision_points, label_parameter="heading_dir", ax=ax[1], per_mouse=True, cmap="Set2")
plotting.pairplot_average_decision_point(decision_points, label_parameter="velocity_x", ax=ax[2], per_mouse=True, cmap="Set2")
plotting.pairplot_average_decision_point(decision_points, label_parameter="head_angle", ax=ax[3], per_mouse=True, cmap="Set2")
plotting.pairplot_average_decision_point(decision_points, label_parameter="local_tortuosity", ax=ax[4], per_mouse=True, cmap="Set2")

plt.tight_layout()

In [ ]:
# bias?
fig, ax = plt.subplots(1, 1)
plotting.plot_rate(
    df=big_df,
    label_x="trial_left_choice",
    per_aperture=True,
    ax=ax,
    cmap="Set2",
    per_mouse=True
)
ax.set_ylim(0, 1)
ax.set_ylabel("P(Left)")
ax.hlines(0.5, xmin=-0.25, xmax=len(big_df.aperture.unique())-0.75, linestyles="dashed", colors="k")

## USE THE DECISION VARIABLE AS AUX VAR FOR CEBRA

In [124]:
import cebra

In [ ]:
df_model

In [125]:
cebra_data = [
    'x',
    'y',
    'velocity_x',
    'velocity_y',
    'heading_dir_sin',
    'heading_dir_cos',
    'head_angle_sin',
    'head_angle_cos',
    'trial_tortuosity',
    'local_tortuosity_1',
    'trial_duration',
    'aperture',
    'trial_rewarded',
]

In [126]:
cebra_labels = [
    'trial_length',
    "proba_left"
]

In [127]:
import sklearn.model_selection
import sklearn.preprocessing

In [128]:
def get_data(df, cebra_data, cebra_label, scale_data=True):
    
    data = df[cebra_data].values
    labels = df[cebra_label].values

    if scale_data:
        data = sklearn.preprocessing.StandardScaler().fit_transform(data)

    return data, labels

In [139]:
model = cebra.CEBRA(device="cuda_if_available", verbose=True, max_iterations=100)

In [ ]:
df_model[df_model.dataset==session_to_plot]

In [ ]:
model.fit(*get_data(df_model[df_model.dataset==session_to_plot], cebra_data=cebra_data, cebra_label=cebra_labels))